In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

In [3]:
# select categorical columns
cat_cols = ["region", "country", "category", "medicine", "age_group"]

In [4]:
# encode categories into numbers
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [5]:
# input columns
x = df[cat_cols].values
# target column
y = df["units_sold"].values

In [6]:
# split data
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print("train shape:", x_train.shape)
print("test shape:", x_test.shape)

train shape: (142392, 5)
test shape: (35598, 5)


In [13]:
# create input data for each column
# tensorflow needs each categorical column separately

train_inputs = {}
test_inputs = {}

for i, col in enumerate(cat_cols):
    train_inputs[col] = x_train[:, i]
    test_inputs[col] = x_test[:, i]

In [14]:
# create embedding layers
inputs = []
embeds = []

for col in cat_cols:

# create input for the column
    inp = tf.keras.Input(
        shape=(1,),
        name=col
    )
# count unique values
    n = df[col].nunique()

# create embedding layer
# output_dim=4 means each category gets 4 numbers
    emb = tf.keras.layers.Embedding(
        input_dim=n,
        output_dim=4,
        name=col + "_emb"
    )(inp)

# convert embedding into simple vector
    emb = tf.keras.layers.Flatten()(emb)

# save input and embedding
    inputs.append(inp)
    embeds.append(emb)

In [15]:
# combine embeddings from all categorical columns
x = tf.keras.layers.Concatenate()(embeds)

In [18]:
#add neural network layers
# first hidden layer
x = tf.keras.layers.Dense(32,activation="relu")(x)

# second hidden layer
x = tf.keras.layers.Dense(16,activation="relu")(x)
# one output because we predict units sold
output = tf.keras.layers.Dense(1)(x)

In [19]:
model = tf.keras.Model(
    inputs=inputs,
    outputs=output
)

In [21]:
model.compile(optimizer="adam",loss="mse")

# show model details
print("\nmodel summary:")
model.summary()


model summary:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ region (InputLayer)           │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ country (InputLayer)          │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ category (InputLayer)         │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ medicine (InputLayer)         │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ age_group (InputLayer)        │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ region_emb (Embedding)        │ (None, 1, 4)              │              32 │ region[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ country_emb (Embedding)       │ (None, 1, 4)              │              76 │ country[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ category_emb (Embedding)      │ (None, 1, 4)              │              20 │ category[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ medicine_emb (Embedding)      │ (None, 1, 4)              │              40 │ medicine[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ age_group_emb (Embedding)     │ (None, 1, 4)              │              20 │ age_group[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_5 (Flatten)           │ (None, 4)                 │               0 │ region_emb[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_6 (Flatten)           │ (None, 4)                 │               0 │ country_emb[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_7 (Flatten)           │ (None, 4)                 │               0 │ category_emb[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_8 (Flatten)           │ (None, 4)                 │               0 │ medicine_emb[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_9 (Flatten)           │ (None, 4)                 │               0 │ age_group_emb[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_1 (Concatenate)   │ (None, 20)                │               0 │ flatten_5[0][0],           │
│                               │                           │                 │ flatten_6[0][0],           │
│                               │                           │                 │ flatten_7[0][0],           │
│                               │                           │               

 Total params: 3,549 (13.86 KB)

 Trainable params: 3,549 (13.86 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
#train the model
print("\ntraining model...")

history = model.fit(
    train_inputs,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)



training model...
Epoch 1/5
3560/3560 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 122466.2188 - val_loss: 106510.9219
Epoch 2/5
3560/3560 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 117453.7344 - val_loss: 104670.6484
Epoch 3/5
3560/3560 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 117467.1250 - val_loss: 104659.8438
Epoch 4/5
3560/3560 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 117479.2109 - val_loss: 104690.4297
Epoch 5/5
3560/3560 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 117423.6953 - val_loss: 104646.7969


In [24]:
#make predictions
pred = model.predict(test_inputs)

1113/1113 ━━━━━━━━━━━━━━━━━━━━ 1s 879us/step


In [25]:
mse = mean_squared_error(y_test,pred)
print("\nmean squared error:", mse)


mean squared error: 116549.4140625


In [26]:
# get learned embeddings
print("\nlearned embeddings:")

for col in cat_cols:

    layer = model.get_layer(col + "_emb")
    weights = layer.get_weights()[0]
    print("\n", col)
    print(weights[:5])


learned embeddings:

 region
[[ 0.24682558  0.21063629 -0.26455846 -0.1627344 ]
 [ 0.28166148  0.26846784 -0.28042436 -0.26419163]
 [ 0.22242399  0.2198486  -0.260247   -0.20152779]
 [ 0.29842395  0.29964525 -0.2925894  -0.19058427]
 [ 0.26700342  0.2446631  -0.26275063 -0.2171083 ]]

 country
[[-0.08256388  0.00353502 -0.13424455 -0.18397681]
 [-0.12546982 -0.10603664 -0.18004012 -0.1090448 ]
 [-0.1540201  -0.16171736 -0.14505976 -0.13880895]
 [-0.15074503 -0.10782801 -0.1729987  -0.1898283 ]
 [-0.17942668 -0.11304687 -0.20716365 -0.13991496]]

 category
[[ 0.06101219 -0.37229887 -0.2430514  -0.31036568]
 [ 0.24295834 -0.37592623 -0.34080455 -0.31088474]
 [ 0.19017968 -0.30664077 -0.29873016 -0.34738103]
 [ 0.10236868 -0.34710625 -0.31619462 -0.31008658]
 [ 0.15421635 -0.31580204 -0.30099866 -0.34075248]]

 medicine
[[ 0.16611898  0.25419953  0.19438913 -0.18658039]
 [ 0.14866817  0.24307033  0.18579037 -0.14744613]
 [ 0.19473854  0.26713842  0.20789501 -0.17037772]
 [ 0.18641265  0.

In [27]:
'''the categorical data was converted into useful numerical embeddings using a neural network. 
these embeddings helped the model learn relationships between different categories and predict units_sold. 
this shows that deep learning embeddings can improve the handling of categorical data in machine learning.'''

'the categorical data was converted into useful numerical embeddings using a neural network. \nthese embeddings helped the model learn relationships between different categories and predict units_sold. \nthis shows that deep learning embeddings can improve the handling of categorical data in machine learning.'